# 07.6 — LoRA: Low-Rank Adaptation

**Problem:** Fine-tuning a 7B parameter LLM requires updating 7B gradients → needs 7B×4 bytes = 28GB of optimizer states alone (Adam stores mean + variance). Unaffordable.

**LoRA insight (Hu et al., 2021):** Pre-trained weight updates have low intrinsic rank. Instead of updating W ∈ ℝ^{d×k} directly, learn two small matrices:
```
W + ΔW = W + B·A
where A ∈ ℝ^{r×k}, B ∈ ℝ^{d×r}, r ≪ min(d,k)
```
Only A and B are trained. W stays frozen.

**Parameter savings:** For W ∈ ℝ^{4096×4096} with r=8:
- Full: 4096² = 16.8M params
- LoRA: 2×4096×8 = 65k params → **256× reduction**

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np

# ---- LoRA implementation ----

class LoRALinear(nn.Module):
    """
    Drop-in replacement for nn.Linear with LoRA.
    
    Forward: y = xW^T + x(BA)^T * (alpha/r)
    
    alpha/r is the scaling factor.
    - alpha is typically set equal to r (scaling factor = 1)
    - or alpha=2r, alpha=16, etc. — hyperparameter
    
    Initialization:
    - A: random Gaussian (kaiming uniform in practice)
    - B: zeros — so ΔW = BA = 0 at initialization
      This means the model starts identical to the pre-trained model.
    """
    def __init__(self, in_features: int, out_features: int, 
                 r: int = 4, lora_alpha: int = 4,
                 lora_dropout: float = 0.0, bias: bool = True):
        super().__init__()
        self.r = r
        self.lora_alpha = lora_alpha
        self.scaling = lora_alpha / r
        
        # Frozen pre-trained weight
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        self.weight.requires_grad = False  # FROZEN
        
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features))
            self.bias.requires_grad = False  # FROZEN
        else:
            self.bias = None
        
        if r > 0:
            # LoRA matrices — TRAINABLE
            self.lora_A = nn.Parameter(torch.empty(r, in_features))
            self.lora_B = nn.Parameter(torch.zeros(out_features, r))
            nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
            # lora_B initialized to zero → ΔW = 0 at start ✓
            
            self.lora_dropout = nn.Dropout(lora_dropout)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Base output (frozen)
        out = F.linear(x, self.weight, self.bias)
        
        # LoRA adapter: x -> A -> B -> scale
        # (lora_B @ lora_A) is the low-rank ΔW, applied to x
        if self.r > 0:
            lora_out = F.linear(
                F.linear(self.lora_dropout(x), self.lora_A),  # [*, r]
                self.lora_B                                    # [out, r]
            ) * self.scaling
            out = out + lora_out
        return out
    
    def merge_weights(self):
        """
        At inference time: merge LoRA into the base weight.
        W_merged = W + (B @ A) * scaling
        Then LoRA is just a regular Linear — no overhead!
        """
        if self.r > 0:
            with torch.no_grad():
                delta_W = (self.lora_B @ self.lora_A) * self.scaling
                self.weight.data += delta_W
                # Zero out LoRA to avoid double-counting
                self.lora_A.data.zero_()
                self.lora_B.data.zero_()


# Verify LoRA has zero effect at initialization
lora_lin = LoRALinear(64, 64, r=4)
x = torch.randn(2, 10, 64)
out = lora_lin(x)
# Without LoRA
out_base = F.linear(x, lora_lin.weight, lora_lin.bias)
print(f"Max diff at init (should be ~0): {(out - out_base).abs().max().item():.8f}")
print(f"  -> LoRA B=0 at init, so ΔW=0. Model starts identical to base.")

# Parameter count
d = 64
r = 4
full_params = d * d
lora_params = 2 * d * r
print(f"\nFull fine-tune: {full_params:,} params")
print(f"LoRA (r={r}):   {lora_params:,} trainable params ({lora_params/full_params:.1%} of full)")

In [ ]:
# ---- Apply LoRA to a Transformer model ----
# Where to inject LoRA?
# Original paper: Q and V projections only.
# QLoRA and later work: all linear layers (Q, K, V, O, gate, up, down in MLP).
# More targets = more capacity = more params.

class LoRATransformerBlock(nn.Module):
    """Transformer block with LoRA on Q and V projections."""
    def __init__(self, d_model, n_heads, lora_r=4, lora_alpha=4):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        # Q and V get LoRA adapters; K and O stay frozen
        self.W_Q = LoRALinear(d_model, d_model, r=lora_r, lora_alpha=lora_alpha)
        self.W_K = nn.Linear(d_model, d_model)  # frozen (no LoRA)
        self.W_V = LoRALinear(d_model, d_model, r=lora_r, lora_alpha=lora_alpha)
        self.W_O = nn.Linear(d_model, d_model)  # frozen
        
        # Freeze K and O (simulate pre-trained frozen weights)
        for p in self.W_K.parameters(): p.requires_grad = False
        for p in self.W_O.parameters(): p.requires_grad = False
        
        self.ff = nn.Sequential(
            nn.Linear(d_model, 4*d_model), nn.GELU(), nn.Linear(4*d_model, d_model)
        )
        # Freeze FF too
        for p in self.ff.parameters(): p.requires_grad = False
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x):
        B, T, D = x.shape
        def split(t): return t.view(B, T, self.n_heads, self.d_k).transpose(1,2)
        Q, K, V = split(self.W_Q(x)), split(self.W_K(x)), split(self.W_V(x))
        scores = (Q @ K.transpose(-2,-1)) / math.sqrt(self.d_k)
        attn = F.softmax(scores, dim=-1)
        out = (attn @ V).transpose(1,2).contiguous().view(B, T, D)
        x = self.norm1(x + self.W_O(out))
        x = self.norm2(x + self.ff(x))
        return x


# Count trainable vs total parameters
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

D = 128
block = LoRATransformerBlock(D, n_heads=4, lora_r=4)
total, trainable = count_params(block)
print(f"LoRA Transformer Block:")
print(f"  Total params:     {total:,}")
print(f"  Trainable params: {trainable:,}  ({trainable/total:.1%})")
print()

# Training only updates lora_A and lora_B
trainable_names = [n for n, p in block.named_parameters() if p.requires_grad]
print(f"Trainable parameter names: {trainable_names}")

In [ ]:
# ---- The rank-r intuition ----
# Why does low-rank work for fine-tuning?

print("=== Low-rank intuition ===")
print()
print("Pre-trained model: W ∈ ℝ^{d×d} (full rank, learned general features)")
print("Fine-tuning task: shift the model toward a specific capability.")
print()
print("Hypothesis (Aghajanyan et al., 2021):")
print("  The 'effective rank' of ΔW during fine-tuning is very low.")
print("  Even if ΔW ∈ ℝ^{4096×4096}, most of its variation lies in r≈8 directions.")
print()
print("Analogy: PCA. A 1000-dimensional dataset of faces might be expressible")
print("in 50 principal components. The 'intrinsic dimensionality' of faces is low.")
print()
print("How to choose r:")
print("  r=1:  extreme compression, works for simple tasks")
print("  r=4:  original paper recommendation")
print("  r=8:  good default for instruction following")
print("  r=16: for complex tasks or larger domain shift")
print("  r=64+: getting close to full fine-tune territory")
print()

# SVD visualization: rank of random vs task-specific updates
np.random.seed(42)
d = 64

# Simulate a 'task update' as sum of a few rank-1 updates
task_rank = 4
delta_W_task = sum(
    np.random.randn(d,1) @ np.random.randn(1,d) * 0.1
    for _ in range(task_rank)
)

# Random noise update (full rank)
delta_W_random = np.random.randn(d, d) * 0.1

# SVD to check effective rank
_, s_task, _ = np.linalg.svd(delta_W_task)
_, s_random, _ = np.linalg.svd(delta_W_random)

# Fraction of variance explained by top-r singular values
for name, sv in [("Task update", s_task), ("Random update", s_random)]:
    for r_check in [4, 8, 16]:
        explained = (sv[:r_check]**2).sum() / (sv**2).sum()
        print(f"  {name} — rank-{r_check:2d} explains {explained:.1%} of variance")

In [ ]:
# ---- QLoRA: LoRA + 4-bit quantization ----
# Fine-tune 70B models on a single 48GB GPU.
# Dettmers et al., 2023.

print("=== QLoRA ===")
print()
print("Key additions over plain LoRA:")
print()
print("1. NF4 (NormalFloat4) quantization of base weights")
print("   - 4-bit quantization (2.5× compression over fp16)")
print("   - Quantization levels chosen to be optimal for normally-distributed weights")
print("   - Reduces memory: 7B params × 4bits/param = 3.5GB (vs 14GB in fp16)")
print()
print("2. Double quantization")
print("   - Quantize the quantization constants themselves")
print("   - Extra 0.37 bits/param savings")
print()
print("3. Paged optimizers")
print("   - Spill optimizer states (Adam moments) to CPU RAM during gradient spikes")
print("   - Prevents OOM on long sequences")
print()
print("4. LoRA adapters in fp16/bf16")
print("   - Despite the base model being 4-bit, LoRA matrices stay in float")
print("   - Dequantize base weight → add LoRA output → result in float")
print()

# Memory estimation for 7B model
params = 7_000_000_000
print("Memory estimates for 7B model:")
print(f"  fp32 weights only:        {params * 4 / 1e9:.1f} GB")
print(f"  fp16 weights + Adam:      {params * (2 + 2*4) / 1e9:.1f} GB  (weights + m + v)")
print(f"  QLoRA (4-bit + LoRA r=8): ~6 GB  (base in NF4, adapters in fp16)")
print()
print("HuggingFace PEFT + bitsandbytes makes this easy:")
print("""
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained('meta-llama/Llama-2-7b-hf', quantization_config=bnb_config)

lora_config = LoraConfig(r=8, lora_alpha=16, target_modules=['q_proj','v_proj'],
                         lora_dropout=0.05, task_type='CAUSAL_LM')
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# trainable params: 4,194,304 || all params: 6,742,609,920 || trainable%: 0.062%
""")

In [ ]:
# ---- Merging LoRA and adapter variants ----

print("=== LoRA variants and alternatives ===")
print()
variants = [
    ("LoRA",     "A and B matrices on linear layers. r=4~16."),
    ("LoRA+",    "Different LRs for A and B (B gets higher LR)."),
    ("DoRA",     "Decompose W into magnitude + direction, LoRA the direction."),
    ("AdaLoRA",  "Adapt rank per layer dynamically using SVD budget."),
    ("IA3",      "Learn 3 scaling vectors (tiny, no matrix multiply)."),
    ("Prefix",   "Prepend learned soft tokens to attention keys/values."),
    ("Prompt",   "Prepend learned embeddings to input (no weight change)."),
]
for name, desc in variants:
    print(f"  {name:10}: {desc}")

print()
print("Merging after training:")
print("  W_merged = W + (B @ A) * (alpha/r)")
print("  Merged model = identical speed to base (LoRA overhead removed)")
print("  Unmerged = slightly slower (two forward passes per layer)")
print()
print("Multiple adapters:")
print("  Train separate LoRA for task A, task B, task C.")
print("  At serving time: hot-swap adapters on the same frozen base.")
print("  LoRAX, vLLM both support this natively.")

# Demo: merge LoRA weights
lora_lin2 = LoRALinear(64, 64, r=4, lora_alpha=8)
x = torch.randn(1, 64)
out_before = lora_lin2(x).detach()
lora_lin2.merge_weights()
out_after = lora_lin2(x).detach()
print(f"\nPost-merge output difference: {(out_before - out_after).abs().max().item():.8f}")
print("(Should be ~0 — merged weights produce identical output)")

## Summary

**LoRA = W + BA where B starts at zero, A is random. Only train A and B.**

**Why it works:** Fine-tuning updates have low intrinsic rank. A few directions in weight space account for task-specific adaptation.

**QLoRA = LoRA + 4-bit base weights.** Fine-tune 7B in 6GB, 70B in ~48GB.

**Merging:** After training, absorb LoRA into W — zero inference overhead. Multiple LoRA adapters can be hot-swapped on the same base model for multi-task serving.

**What breaks it:** Very large domain shifts (training data drastically different from pre-training). In that case, full fine-tuning or continued pre-training is needed.

---
**Next:** 07.7 — Prompting and Few-shot Learning